# Identification seed point par FTD

In [ ]:
# Chargement des librairies
import numpy as np
import matplotlib.pyplot as plt
import os
import plotly.graph_objects as go
import nibabel as nib
from skimage.feature import canny
from skimage.transform import hough_circle, hough_circle_peaks
from skimage.draw import circle_perimeter


In [ ]:
# Let's start with a single image
# Visualisation des images d'un patient

training_path = "database/training"

# Visualisation des images d'un patient
numero_patient = 1

# Load the images
image_4d = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_4d.nii.gz")
image_01 = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_frame01.nii.gz")
image_01_gt = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_frame01_gt.nii.gz")
image_12 = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_frame12.nii.gz")
image_12_gt = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_frame12_gt.nii.gz")

print(image_4d.shape) # (216, 256, 10, 30) for patient 1, corresponding to 10 slices and 30 time frames
r, c, nb_frames, duration = image_4d.shape

print("Images loaded")

In [ ]:
# Plot the images

layer = 4

fig, axs = plt.subplots(2, 2, figsize=(10, 10))
axs[0, 0].imshow(image_01.dataobj[:, :, layer], cmap="gray")
axs[0, 0].set_title(f"Frame 01, layer {layer}")
axs[1, 0].imshow(image_01_gt.dataobj[:, :, layer], cmap="gray")
axs[1, 0].set_title(f"Frame 01 GT, layer {layer}")
axs[0, 1].imshow(image_12.dataobj[:, :, layer], cmap="gray")
axs[0, 1].set_title(f"Frame 12, layer {layer}")
axs[1, 1].imshow(image_12_gt.dataobj[:, :, layer], cmap="gray")
axs[1, 1].set_title(f"Frame 12 GT, layer {layer}")
plt.show()


In [ ]:
# Let's perform a fourrier transform on the first image (time axis)
image = image_4d.get_fdata()

# Keep only the first layer
image = image[:, :, layer, :]
print(image.shape)

# Perform the fourrier transform
image_fft = np.fft.fft(image, axis=2)

print(image_fft.shape)

# Plot the fourrier transform
fig, axs = plt.subplots(1, 10, figsize=(20, 2))
for i in range(10):
    axs[i].imshow(np.abs(image_fft[:, :, i]), cmap="gray")
    axs[i].set_title(f"Frame {i}")
plt.show()

# Only keep the first harmonic
image_fft_h1 = abs(image_fft[:, :, 1])

print(image_fft_h1.shape)

# Keep the highest values
image_fft_h1 = image_fft_h1 > np.percentile(image_fft_h1, 98)

# Plot the average fourrier transform
plt.imshow(image_fft_h1, cmap="gray")
plt.title("Fourrier transform filter (threshold 95%)")
plt.show()

In [ ]:
# Perform the hough transform and plot all found circles.

# Edge detection 
edges = canny(image_fft_h1, sigma=1, low_threshold=0.1, high_threshold=1)

plt.imshow(edges, cmap="gray")
plt.title("Edges found")
plt.show()

# Hough transform
hough_radii = np.arange(10, 20, 2)
hough_res = hough_circle(edges, hough_radii)
accums, cx, cy, radii = hough_circle_peaks(hough_res, hough_radii, total_num_peaks=10)

print(f"Found {len(cx)} circles")

fig, ax = plt.subplots()
ax.imshow(edges, cmap="gray")
for center_y, center_x, radius in zip(cy, cx, radii):
    circle = plt.Circle((center_x, center_y), radius, color='r', fill=False)
    ax.add_artist(circle)
ax.set_title("Circles found")
plt.show()

#Plot the average center of the circles
center_x_avg = np.mean(cx)
center_y_avg = np.mean(cy)
plt.imshow(edges, cmap="gray")
plt.scatter(center_x_avg, center_y_avg, color="r")
plt.title("Chosen center point")
plt.show()

In [ ]:
def compute_fft_avg(image_4d):
    # Perform the fourrier transform on the first image (time axis)
    image = image_4d.get_fdata()

    # Perform the fourrier transform
    image_fft = np.fft.fft(image, axis=2)

    # only keep the first harmonic
    image_fft_h1 = abs(image_fft[:, :, 1])

    # Keep the highest values
    image_fft_h1 = image_fft_h1 > np.percentile(image_fft_h1, 98)
    return image_fft_h1

def find_seed_location(image_fft_h1):
    # Edge detection 
    edges = canny(image_fft_h1, sigma=1, low_threshold=0.1, high_threshold=1)

    # Hough transform
    hough_radii = np.arange(10, 20, 2)
    hough_res = hough_circle(edges, hough_radii)
    accums, cx, cy, radii = hough_circle_peaks(hough_res, hough_radii, total_num_peaks=10)

    #Plot the average center of the circles
    center_x_avg = np.mean(cx)
    center_y_avg = np.mean(cy)
    return center_x_avg, center_y_avg

print("Function defined")

In [ ]:
# Let's test with a few patients
patient_numbers = [1, 2, 3, 4, 5]

for patient in patient_numbers:
    image_4d = nib.load(f"{training_path}/patient{patient:03d}/patient{patient:03d}_4d.nii.gz")
    image_fft_avg = compute_fft_avg(image_4d)
    center_x_avg, center_y_avg = find_seed_location(image_fft_avg)
    print(f"Patient {patient} - Center: ({center_x_avg}, {center_y_avg})")